# Preprocessing and Baseline
[Previous Notebook](dataset_audit.ipynb) | [Next Notebook (TBD)]()

Preprocessing the dataset is needed for this dataset as it include categorical features

In [43]:
%store -r Teen_Mental_Health_Dataset

data = Teen_Mental_Health_Dataset.copy() # type: ignore
display(data.head())

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,not depressed
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,not depressed
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,not depressed
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,not depressed
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,not depressed


## Features and Target
To split the data into a **training set** and a **testing set**, I first need to split it into a **feature set** `X` and a **target set** `y`.

In [ ]:
X = data.drop(columns=['depression_label'])
y = data['depression_label']

## Training and Testing
Now that the data is split into a a **feature set** `X` and a **target set** `y`, I can split these into:
- **Training feature set** `X_train`
- **Testing feature set** `X_test`
- **Training target set** `y_train`
- **Testing target set** `y_test`

Using a `test_size=0.25`, i will split the data into 75% training and 25% testing. Since the dataset is so imbalanced, I chose a little bit higher percentage of training data than recommended in the slides to both mitigate overfitting and also to give the training set more opportunities to test positive target values.

I am using `stratify=y` to stratify `depression_label` so that the splits have a similar proportion of positive and negative target values

**N.B:** The data **MUST** be split before any machine learning practices are done to avoid leaking test information into the model.

In [46]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## Encoding Categorical features to numerical values.
I am using `OneHotEncoder` to encode the categorical features `gender` and `platform_usage`. This will create a new column for each unique value, i.e. "male", "female", "instagram", etc. 

I am using `OrdinalEncoder` to encode the categorical features `social_interaction_level`. This will convert the values from low, medium, high to a range (0-2)

In [47]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

column_transformer = ColumnTransformer(
    transformers=[
        (
            "onehot_encoder",
            OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
            ['gender', 'platform_usage']
        ),
        (
            "ordinal_encoder",
            OrdinalEncoder(),
            ['social_interaction_level'],
        )
    ],
    remainder="passthrough"
)

# Baseline Model

For a baseline, i will use a simple `LogisticRegression` model.

In [48]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000, random_state=42);

## Pipeline

To keep preprocessing coherent and avoid leakage, I'll use scikitlearn's pipeline API.

In [49]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("column transformer", column_transformer),
    ("baseline", baseline_model)
])

%store pipeline

Stored 'pipeline' (Pipeline)


# Cross Validation

In [50]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring="f1_macro",
)

print(f"Macro-F1 scores: {scores}")
print(f"Mean macro-F1: {scores.mean():.3f}")
print(f"95% range: +/- {scores.std() * 2:.3f}")

Macro-F1 scores: [0.66101695 0.78146499 0.94301994 0.79428571 0.82905983]
Mean macro-F1: 0.802
95% range: +/- 0.181


# Predictions and Metrics

In [51]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
import pandas as pd

model = pipeline.fit(X_train, y_train)
predictions = model.predict(X_test)

display(pd.DataFrame(confusion_matrix(y_test, predictions, labels=model.classes_), index=model.classes_, columns=model.classes_))
print("Accuracy:", accuracy_score(y_test, predictions))
print("Balanced accuracy:", balanced_accuracy_score(y_test, predictions))
print("Macro-F1:", f1_score(y_test, predictions, average="macro"))
print("\nClassification report:\n")
print(classification_report(y_test, predictions, zero_division=0))

,depressed,not depressed
depressed,4,4
not depressed,1,291


Accuracy: 0.9833333333333333
Balanced accuracy: 0.7482876712328768
Macro-F1: 0.8034333639103657

Classification report:

               precision    recall  f1-score   support

    depressed       0.80      0.50      0.62         8
not depressed       0.99      1.00      0.99       292

     accuracy                           0.98       300
    macro avg       0.89      0.75      0.80       300
 weighted avg       0.98      0.98      0.98       300



## [Go to the next notebook (TBD)]()